# Single-Agent Pipeline

A smart assistant that understands a query, decides what kind of task it is, routes it to the right tool, and returns a structured JSON response.

The agent handles several cases: math goes to a calculator, keyword requests go to a keyword extractor, text transformation goes to a text tool, and unit conversions go to a converter. Anything else gets a general response. It also logs what it does and handles errors so one bad query does not break the run.

## Tools

Each tool does one job and returns a clean result.

In [1]:
# 🛠️ Tool 1: Calculator
def calculator(expression: str) -> str:
    """Evaluate a simple math expression."""
    try:
        allowed = set("0123456789+-*/(). %")
        if not set(expression).issubset(allowed):
            return "Error: expression has unsupported characters"
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ Tool 2: Keyword Extractor
def extract_keywords(text: str) -> list:
    """Pull out a few keywords from the text."""
    try:
        stopwords = {"from", "this", "that", "with", "what", "about", "your", "have", "will"}
        words = text.split()
        keywords = []
        for w in words:
            cleaned = w.lower().strip(".,!?")
            if len(cleaned) > 4 and cleaned not in stopwords and cleaned not in keywords:
                keywords.append(cleaned)
        return keywords[:5]
    except Exception:
        return []

In [3]:
# 🛠️ Tool 3: Text Transformer (reverse / uppercase / lowercase)
def transform_text(text: str, mode: str) -> str:
    """Reverse text, or change its case."""
    try:
        if mode == "reverse":
            return text[::-1]
        elif mode == "upper":
            return text.upper()
        elif mode == "lower":
            return text.lower()
        else:
            return "Error: unknown text mode"
    except Exception:
        return "Error in text transformation"

In [4]:
# 🛠️ Tool 4: Unit Converter (km<->miles, kg<->lb, c<->f)
def convert_unit(value: float, unit: str) -> str:
    """Convert between a few common units."""
    try:
        unit = unit.lower()
        if unit == "km":
            return f"{value * 0.621371:.2f} miles"
        elif unit == "miles":
            return f"{value / 0.621371:.2f} km"
        elif unit == "kg":
            return f"{value * 2.20462:.2f} lb"
        elif unit == "lb":
            return f"{value / 2.20462:.2f} kg"
        elif unit == "c":
            return f"{value * 9/5 + 32:.2f} F"
        elif unit == "f":
            return f"{(value - 32) * 5/9:.2f} C"
        else:
            return "Error: unsupported unit"
    except Exception:
        return "Error in conversion"

## Routing

The agent decides which path a query takes based on clear signals in the text. Keeping the rules explicit makes the routing easy to follow and easy to extend with new tools.

In [5]:
import re
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")
log = logging.getLogger("agent")
def route(query: str) -> str:
    q = query.lower()
    if "calculate" in q or "compute" in q or re.search(r"\d\s*[\+\-\*/]\s*\d", query):
        return "calculation"
    if "keyword" in q or "extract" in q:
        return "keywords"
    if "reverse" in q or "uppercase" in q or "lowercase" in q:
        return "text"
    if "convert" in q or re.search(r"\d+\s*(km|miles|kg|lb|c|f)\b", q):
        return "conversion"
    return "general"

## The Agent

This ties everything together. It logs the query, routes it, calls the right tool, and returns a JSON-style dictionary with the task type and result. Any unexpected error is caught and returned as an error type instead of crashing.

In [6]:
def agent(query: str) -> dict:
    try:
        task_type = route(query)
        log.info(f"query routed as: {task_type}")

        if task_type == "calculation":
            expr = re.sub(r"[a-zA-Z]+", "", query).strip().strip(":?").strip()
            result = calculator(expr if expr else query)
            return {"type": "calculation", "result": result}

        elif task_type == "keywords":
            text = re.sub(r"(?i)extract keywords from", "", query).strip()
            result = extract_keywords(text if text else query)
            return {"type": "keywords", "result": result}

        elif task_type == "text":
            q = query.lower()
            mode = "reverse" if "reverse" in q else "upper" if "uppercase" in q else "lower"
            text = re.sub(r"(?i)(reverse|convert to uppercase|convert to lowercase|uppercase|lowercase|the text|text)", "", query).strip()
            result = transform_text(text if text else query, mode)
            return {"type": "text", "result": result}

        elif task_type == "conversion":
            match = re.search(r"(\d+\.?\d*)\s*(km|miles|kg|lb|c|f)\b", query.lower())
            if match:
                value = float(match.group(1))
                unit = match.group(2)
                result = convert_unit(value, unit)
            else:
                result = "Error: could not read value and unit"
            return {"type": "conversion", "result": result}
        else:
            return {"type": "general", "result": "I can do math, pull out keywords, transform text, or convert units. Try one of those."}

    except Exception as e:
        log.info(f"error: {e}")
        return {"type": "error", "result": str(e)}

## Expected Output Format

Every response is a dictionary with two keys:

```
{
  "type": "calculation / keywords / text / conversion / general / error",
  "result": ...
}
```

## Test Cases

In [7]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "compute 144 / 12",
    "Calculate 10 / 0",
    "Reverse the text hello world",
    "Convert to uppercase machine learning",
    "Convert 10 km to miles",
    "Convert 75 f to celsius"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 55)

2026-06-28 19:25:41,525 | query routed as: calculation
2026-06-28 19:25:41,526 | query routed as: keywords
2026-06-28 19:25:41,526 | query routed as: general
2026-06-28 19:25:41,526 | query routed as: calculation
2026-06-28 19:25:41,527 | query routed as: calculation
2026-06-28 19:25:41,527 | query routed as: text
2026-06-28 19:25:41,527 | query routed as: text
2026-06-28 19:25:41,528 | query routed as: conversion
2026-06-28 19:25:41,528 | query routed as: conversion


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
-------------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'transforming', 'industries']}
-------------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'I can do math, pull out keywords, transform text, or convert units. Try one of those.'}
-------------------------------------------------------
Query: compute 144 / 12
Response: {'type': 'calculation', 'result': '12.0'}
-------------------------------------------------------
Query: Calculate 10 / 0
Response: {'type': 'calculation', 'result': 'Error in calculation'}
-------------------------------------------------------
Query: Reverse the text hello world
Response: {'type': 'text', 'result': 'dlrow olleh'}
---------------------------------------------

## Interactive Mode

Type your own queries. Enter `exit` to stop.

In [9]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ").strip()
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))
    print("-" * 55)

Enter query (type 'exit' to stop):  Calculate 20 + 5


2026-06-28 19:26:38,613 | query routed as: calculation


Response: {'type': 'calculation', 'result': '25'}
-------------------------------------------------------


Enter query (type 'exit' to stop):  Extract keywords from Artificial Intelligence is transforming industries


2026-06-28 19:26:45,443 | query routed as: keywords


...

Response: {'type': 'conversion', 'result': '6.21 miles'}
-------------------------------------------------------


Enter query (type 'exit' to stop):  Convert 75 f to celsius


2026-06-28 19:27:48,293 | query routed as: conversion


Response: {'type': 'conversion', 'result': '23.89 C'}
-------------------------------------------------------


Enter query (type 'exit' to stop):  exit


## Notes

The agent covers the required pieces: routing by intent, tool integration, structured JSON output, and error handling.

For the bonus, all three items are done. Routing is improved (it catches arithmetic written as symbols and unit values, not just keywords). Logging shows how each query was routed. And two new tools were added beyond the original calculator and keyword extractor: a text transformer (reverse, uppercase, lowercase) and a unit converter (km, miles, kg, lb, celsius, fahrenheit).